In [134]:
import pandas as pd
import numpy as np
import re
DATASET = pd.read_csv('Comune-di-Milano-Servizi-alla-persona-parrucchieri-estetisti(in).csv',sep=';',encoding='unicode_escape')
DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
0,NaN,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0
1,NaN,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0
2,NaN,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0
3,NaN,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN
4,NaN,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0
...,...,...,...,...,...,...,...,...,...,...
3904,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,VIA,SARPI FRA' PAOLO,1,7210.0,1,NaN,NaN,NaN
3905,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),CSO,DI PORTA TICINESE,4,541.0,1,NaN,NaN,65.0
3906,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN
3907,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,NaN,640.0,1,NaN,NaN,NaN


Consistency

In [135]:
# 1. Convert column to appropriate data type and handle NaN
# Now convert to numeric
DATASET['Civico'] = pd.to_numeric(DATASET['Civico'], errors='coerce')
DATASET['Civico'] = DATASET['Civico'].astype(pd.Int64Dtype())
DATASET['ZD'] = pd.to_numeric(DATASET['ZD'], errors='coerce')
DATASET['ZD'] = DATASET['ZD'].astype(pd.Int64Dtype())
DATASET['Superficie lavorativa'] = pd.to_numeric(DATASET['Superficie lavorativa'], errors='coerce')
DATASET['Superficie altri usi'] = pd.to_numeric(DATASET['Superficie altri usi'], errors='coerce')

# Ensure non-empty strings
DATASET['Tipo esercizio pa'] = DATASET['Tipo esercizio pa'].fillna("")

DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
0,,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0
1,,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0
2,,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0
3,,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN
4,,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0
...,...,...,...,...,...,...,...,...,...,...
3904,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,VIA,SARPI FRA' PAOLO,1,7210.0,1,NaN,NaN,NaN
3905,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),CSO,DI PORTA TICINESE,4,541.0,1,NaN,NaN,65.0
3906,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN
3907,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN


# Data wrangling

## Colonna Tipo esercizio pa

In [136]:
DATASET.rename(columns={'Prevalente':'Attività primaria','ZD':'Municipio'}, inplace=True)

In [137]:
#Standardization for "Tipo esercizio pa"

# Divido la colonna 'Tipo esercizio pa' in più righe
DATASET = DATASET.assign(**{'Tipo esercizio pa': DATASET['Tipo esercizio pa'].str.split(';')}).explode('Tipo esercizio pa')

replacements = {
    'TIPO A - REG.2003': 'Estetista',
    'TIPO A ESTETICA MANUALE': 'Estetista',
    'TIPO B CENTRO DI ABBRONZATURA': 'Centro abbronzatura',
    'TIPO C TRATT.ESTETICI DIMAGRIM': 'Trattamento estetico dimagrimento',
    'TIPO D ESTET.APPAR.ELETTROMECC': 'Estetica con apparecchiature',
    'TIPO A-B-C-D': 'Estetista;Centro abbronzatura;Trattamento estetico dimagrimento;Estetica con apparecchiature',
    'TIPO A-B-C-D in profumeria': 'Estetista;Centro abbronzatura;Trattamento estetico dimagrimento;Estetica con apparecchiature',
    'BARBIERE': 'Barbiere',
    'ACCONCIATORE': 'Acconciatore',
    'esecuzione di tatuaggi e piercing': 'Esecuzione di tatuaggi e piercing'
}

for old_value, new_value in replacements.items():
    DATASET.loc[DATASET['Tipo esercizio pa'] == old_value, 'Tipo esercizio pa'] = new_value

DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attività primaria,Superficie altri usi,Superficie lavorativa
0,,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0
1,,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0
2,,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0
3,,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN
4,,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0
...,...,...,...,...,...,...,...,...,...,...
3906,Centro abbronzatura,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN
3907,Estetica con apparecchiature,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN
3907,Trattamento estetico dimagrimento,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN
3907,Centro abbronzatura,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN


In [138]:
# Divido la colonna 'Tipo esercizio pa' in più righe per finire la standardizzazione
DATASET = DATASET.assign(**{'Tipo esercizio pa': DATASET['Tipo esercizio pa'].str.split(';')}).explode('Tipo esercizio pa')
DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attività primaria,Superficie altri usi,Superficie lavorativa
0,,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0
1,,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0
2,,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0
3,,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN
4,,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0
...,...,...,...,...,...,...,...,...,...,...
3906,Centro abbronzatura,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN
3907,Estetica con apparecchiature,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN
3907,Trattamento estetico dimagrimento,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN
3907,Centro abbronzatura,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN


In [139]:
# Verifica valori unici PRIMA della standardizzazione
DATASET['Tipo esercizio pa'].value_counts()

Tipo esercizio pa
Estetista                            1245
Parrucchiere per signora             1214
Acconciatore                          907
Centro abbronzatura                   693
Parrucchiere per uomo                 518
Trattamento estetico dimagrimento     226
Estetica con apparecchiature          162
Centro massaggi                       139
Parrucchiere misto                    105
                                       31
Centro benessere                       29
Esecuzione di tatuaggi e piercing      24
Pedicure estetico                      22
Manicure                                3
Barbiere                                2
Estetista in profumeria                 2
 (z.d. 9)                               1
Truccatore                              1
Name: count, dtype: int64

## Ubicazione

In [140]:
#Replacement for Ubicazione
DATASET['Ubicazione'] = DATASET['Ubicazione'].str.replace('num', 'N')

In [141]:
# Splitting the "Ubicazione" column on "N." to separate address and addition data
split_column = DATASET['Ubicazione'].str.split(r"\bN\.\s*", n=1, expand=True)
split_column.columns = ['Ubicazione_effettiva', 'Ubicazione_data']

# Add split columns to DATASET
DATASET = pd.concat([DATASET, split_column], axis=1)

# Splitting the "Ubicazione_data" column on "(" to separate civico and z.d.
split_column_2 = DATASET['Ubicazione_data'].str.split("(", n=1, expand=True)
split_column_2.columns = ['Civico_to_check', 'z.d._to_check']

# Add to original database
DATASET = pd.concat([DATASET, split_column_2], axis=1)

# Remove ";" from "Civico_to_check"
DATASET['Civico_to_check'] = DATASET['Civico_to_check'].str.replace(";", "", regex=False)

# Remove the closing parenthesis ")" and the z.d. description from the "z.d._to_check" column
DATASET["z.d._to_check"] = DATASET["z.d._to_check"].str.replace("z.d.", "", regex=False)
DATASET["z.d._to_check"] = DATASET["z.d._to_check"].str.replace(")", "", regex=False)

# Splitting the "Ubicazione_effettiva" column on the first " " to separate "Tipo via" and "Via"
split_column_3 = DATASET['Ubicazione_effettiva'].str.split(" ", n=1, expand=True)
split_column_3.columns = ['Tipo via_to_check', 'Via_to_check']

# Add to original database
DATASET = pd.concat([DATASET, split_column_3], axis=1)

DATASET = DATASET.drop(['Ubicazione_effettiva', 'Ubicazione_data'], axis=1)
DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attività primaria,Superficie altri usi,Superficie lavorativa,Civico_to_check,z.d._to_check,Tipo via_to_check,Via_to_check
0,,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0,10,6,LGO,DEI GELSOMINI
1,,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0,3,9,PZA,FIDIA
2,,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0,10,5,VIA,ADIGE
3,,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN,9,1,VIA,BARACCHINI FLAVIO
4,,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0,12,4,VIA,BERGAMO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3906,Centro abbronzatura,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN,2,9,VIA,CANDOGLIA
3907,Estetica con apparecchiature,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,002a,1,VIA,NIRONE
3907,Trattamento estetico dimagrimento,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,002a,1,VIA,NIRONE
3907,Centro abbronzatura,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,002a,1,VIA,NIRONE


In [147]:
# Clean and convert Civico_to_check and z.d._to_check before comparison
def clean_numeric_string(value):
    if pd.isna(value):
        return pd.NA
    value_str = str(value).strip()
    if not value_str:
        return pd.NA
    # Split at first space and take only the numeric part
    value_str = value_str.split(' ')[0]
    # Remove leading zeros but keep at least one digit
    value_str = re.sub(r'^0+(?=\d)', '', value_str)
    if not value_str:
        value_str = "0"
    return value_str

# Apply cleaning to both columns
DATASET['Civico_to_check'] = DATASET['Civico_to_check'].apply(clean_numeric_string)
DATASET['z.d._to_check'] = DATASET['z.d._to_check'].apply(clean_numeric_string)

# Now convert to Int64
DATASET['Civico_to_check'] = pd.to_numeric(DATASET['Civico_to_check'], errors='coerce').astype(pd.Int64Dtype())
DATASET['z.d._to_check'] = pd.to_numeric(DATASET['z.d._to_check'], errors='coerce').astype(pd.Int64Dtype())

# Check if the results are the same
condition = (DATASET['Civico_to_check'] == DATASET['Civico']) & (DATASET['z.d._to_check'] == DATASET['Municipio'])

# Show rows where condition is satisfied
DATASET[~condition]

#TODO: Manca il tipo via e il nome via, ma bisogna modificare un po' de robba (Non so se da fare)

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attività primaria,Superficie altri usi,Superficie lavorativa,Civico_to_check,z.d._to_check,Tipo via_to_check,Via_to_check
210,Acconciatore,codvia 4386 N.008; (z.d. 4),CSO,LODI,72,4068.0,4,NaN,NaN,50.0,8,4,codvia,4386
243,Acconciatore,mm1 duomo codvia 9112 N.000; (z.d. 1),CSO,SEMPIONE,10,7137.0,1,NaN,NaN,NaN,0,1,mm1,duomo codvia 9112
280,Acconciatore,VIA ALTAMURA SAVERIO N. 7 ; (z.d. 5),VIA,ALTAMURA SAVERIO,7,6576.0,7,NaN,NaN,NaN,7,5,VIA,ALTAMURA SAVERIO
368,Acconciatore,VIA CONSOLE MARCELLO N.0181; (z.d. 8),VIA,SILVA GUGLIELMO,49,6400.0,8,NaN,NaN,52.0,181,8,VIA,CONSOLE MARCELLO
429,Acconciatore,VIA FIUGGI N.0121; (z.d. 9),VIA,DE MARTINO EMILIO,1,1693.0,9,NaN,NaN,NaN,121,9,VIA,FIUGGI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3699,Centro abbronzatura,VIA PAGANO MARIO piano rialzato N.03739; (z.d. 1),CSO,VENEZIA,14,238.0,1,NaN,NaN,NaN,3739,1,VIA,PAGANO MARIO piano rialzato
3764,Estetista,VIA SERLIO SEBASTIANO N.0082; (z.d. 4),VIA,MINCIO,3,4157.0,4,NaN,NaN,NaN,82,4,VIA,SERLIO SEBASTIANO
3764,Centro abbronzatura,VIA SERLIO SEBASTIANO N.0082; (z.d. 4),VIA,MINCIO,3,4157.0,4,NaN,NaN,NaN,82,4,VIA,SERLIO SEBASTIANO
3767,Estetista,VIA SOLARI ANDREA N.043; (z.d. 6),VIA,LORENTEGGIO,157,5132.0,6,NaN,NaN,NaN,43,6,VIA,SOLARI ANDREA


# Altro

In [143]:
# Divido la colonna 'Tipo esercizio pa' in più righe
df = DATASET.assign(**{'Tipo esercizio pa': DATASET['Tipo esercizio pa'].str.split(';')}).explode('Tipo esercizio pa')
df

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attività primaria,Superficie altri usi,Superficie lavorativa,Civico_to_check,z.d._to_check,Tipo via_to_check,Via_to_check
0,,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0,10,6,LGO,DEI GELSOMINI
1,,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0,3,9,PZA,FIDIA
2,,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0,10,5,VIA,ADIGE
3,,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN,9,1,VIA,BARACCHINI FLAVIO
4,,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0,12,4,VIA,BERGAMO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3906,Centro abbronzatura,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN,2,9,VIA,CANDOGLIA
3907,Estetica con apparecchiature,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,<NA>,1,VIA,NIRONE
3907,Trattamento estetico dimagrimento,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,<NA>,1,VIA,NIRONE
3907,Centro abbronzatura,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,<NA>,1,VIA,NIRONE


In [144]:
# Cambio carattere
df['Tipo esercizio pa'] = df['Tipo esercizio pa'].str.title()
df

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attività primaria,Superficie altri usi,Superficie lavorativa,Civico_to_check,z.d._to_check,Tipo via_to_check,Via_to_check
0,,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0,10,6,LGO,DEI GELSOMINI
1,,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0,3,9,PZA,FIDIA
2,,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0,10,5,VIA,ADIGE
3,,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN,9,1,VIA,BARACCHINI FLAVIO
4,,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0,12,4,VIA,BERGAMO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3906,Centro Abbronzatura,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN,2,9,VIA,CANDOGLIA
3907,Estetica Con Apparecchiature,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,<NA>,1,VIA,NIRONE
3907,Trattamento Estetico Dimagrimento,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,<NA>,1,VIA,NIRONE
3907,Centro Abbronzatura,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE,<NA>,640.0,1,NaN,NaN,NaN,<NA>,1,VIA,NIRONE


In [145]:
# Controllo prima se i valori nella colonna 'Ubicazione' sono presenti nelle altre colonne
columns_address = ['Tipo via', 'Via', 'Civico', 'ZD']

pattern = r'(\w+)\s+(.+?)\s+(?:N\.|num\.?)\s*(\d+)(?:\s*;?\s*\(z\.d\.\s*(\d+)\))?.*'

# Applica l'estrazione e crea nuove colonne
df_nuove_colonne = df['Ubicazione'].str.extract(pattern)

# Controlla se i valori nelle altre colonne sono uguali a quelli estratti
for i, col in enumerate(columns_address):
    if df_nuove_colonne[i].notnull().any():
        df[f'check_{col}'] = df_nuove_colonne[i] == df[col]
df

# Stampa le righe con False
# TODO: da rivedere
for col in columns_address:
    false_rows = df[df[f'check_{col}'] == False]
    if not false_rows.empty:
        print(f"Discrepanze trovate nella colonna '{col}':")
        print(false_rows[['Ubicazione', col, f'check_{col}']])

KeyError: 'ZD'

In [ ]:
# Rimuovo tutto quello dopo N. nella colonna Ubicazione
df['Ubicazione'] = df['Ubicazione'].str.replace(
    r'\s+(N\.|num\.?)\s*\d+.*', '', regex=True
)

# Rimuovo z.d.
regex_pattern = r'\(z\.d\.\s*\d+\)'
df['Tipo esercizio pa'] = np.where(
    df['Tipo esercizio pa'].astype(str).str.contains(regex_pattern, regex=True),
    np.nan,
    df['Tipo esercizio pa']
)
df

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa,check_Tipo via,check_Via,check_Civico,check_ZD
0,NaN,LGO DEI GELSOMINI,LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0,True,True,True,True
1,NaN,PZA FIDIA,PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0,True,True,True,True
2,NaN,VIA ADIGE,VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0,True,True,True,True
3,NaN,VIA BARACCHINI FLAVIO,VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN,True,True,True,True
4,NaN,VIA BERGAMO,VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3906,Tipo B Centro Di Abbronzatura,VIA CANDOGLIA,VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN,True,True,True,True
3907,Tipo D Estet.Appar.Elettromecc,VIA NIRONE,VIA,NIRONE,NaN,640.0,1,NaN,NaN,NaN,True,True,False,False
3907,Tipo C Tratt.Estetici Dimagrim,VIA NIRONE,VIA,NIRONE,NaN,640.0,1,NaN,NaN,NaN,True,True,False,False
3907,Tipo B Centro Di Abbronzatura,VIA NIRONE,VIA,NIRONE,NaN,640.0,1,NaN,NaN,NaN,True,True,False,False
